In [ ]:
!pip -q install -U transformers accelerate datasets peft trl bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.1 MB/s eta 0:00:00


In [ ]:
!nvidia-smi

Fri Mar 27 09:32:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

torch.cuda.empty_cache()
gc.collect()

model_name = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
model.config.use_cache = True

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

# Run 1 inference with base model

In [ ]:
import pandas as pd
import torch

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

df = pd.read_csv("train_robotics.csv")
row = df.iloc[429]
text = row["x"]

def truncate_text_to_tokens(text, tokenizer, max_doc_tokens=1600):
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    ids = ids[:max_doc_tokens]
    return tokenizer.decode(ids, skip_special_tokens=True), len(ids)

max_ctx = 2048
max_new = 200
max_input = max_ctx - max_new

prefix = "Summarize the following scientific paper excerpt in 3-5 sentences.\n\n"
suffix = "\n\nSummary:\n"

prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
suffix_ids = tokenizer(suffix, add_special_tokens=False)["input_ids"]
budget_for_text = max_input - len(prefix_ids) - len(suffix_ids)

text_ids = tokenizer(text, add_special_tokens=False)["input_ids"][:budget_for_text]
truncated_text = tokenizer.decode(text_ids, skip_special_tokens=True)

prompt = prefix + truncated_text + suffix

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=False,
    padding=False
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
final_summary = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("\n=== FINAL SUMMARY ===\n")
print(final_summary if final_summary else "[EMPTY OUTPUT]")

print("\n=== REFERENCE SUMMARY ===\n")
print(row["summary"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== FINAL SUMMARY ===

This paper proposes a novel approach to integrating deep-reinforcement-learning-based obstacle avoidance with conventional global planners. The authors introduce an "intermediate planner" that generates waypoints more dynamically and flexibly than traditional global planners. They propose two different waypoint generators, including a subsampling approach and a spatial-time-horizon approach, to interconnect DRL-based local planners with global planners. The authors evaluate the performance of their approach against conventional planning systems and demonstrate promising results.

=== REFERENCE SUMMARY ===

Deep Reinforcement Learning has emerged as an efficient dynamic obstacle
avoidance method in highly dynamic environments. It has the potential to
replace overly conservative or inefficient navigation approaches. However, the
integration of Deep Reinforcement Learning into existing navigation systems is
still an open frontier due to the myopic nature of
Deep-Re

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=912df301b3d838ac0897c3659c9767d1e0d16273f37300402357d00021e68f5b
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

scores = scorer.score(row["summary"], final_summary)

rouge_l = scores['rougeL']

print("ROUGE-L:")
print(f"Precision: {rouge_l.precision:.4f}")
print(f"Recall:    {rouge_l.recall:.4f}")
print(f"F1:        {rouge_l.fmeasure:.4f}")

ROUGE-L:
Precision: 0.2895
Recall:    0.1719
F1:        0.2157


# FineTune

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [ ]:
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

model.config.use_cache = False

model.gradient_checkpointing_enable()

model.print_trainable_parameters()

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [ ]:
import sys
import csv
from datasets import Dataset

csv.field_size_limit(sys.maxsize)

train_df = pd.read_csv("train_robotics.csv").dropna(subset=["x", "summary"])
test_df = pd.read_csv("test_robotics.csv", engine="python").dropna(subset=["x", "summary"])

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

max_ctx = 2048
max_new = 200
max_input = max_ctx - max_new

prefix = "Summarize the following scientific paper excerpt in 3-5 sentences.\n\n"
suffix = "\n\nSummary:\n"

prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
suffix_ids = tokenizer(suffix, add_special_tokens=False)["input_ids"]


In [ ]:
def format_example(row):
    x_text = str(row["x"])
    y_text = str(row["summary"]).strip() + tokenizer.eos_token

    y_ids = tokenizer(y_text, add_special_tokens=False)["input_ids"]

    budget_for_x = max_input - len(prefix_ids) - len(suffix_ids) - len(y_ids)

    if budget_for_x < 0:
        y_ids = y_ids[: max(32, max_input // 4)]
        budget_for_x = max_input - len(prefix_ids) - len(suffix_ids) - len(y_ids)

    x_ids = tokenizer(x_text, add_special_tokens=False)["input_ids"][:max(0, budget_for_x)]

    prompt_ids = prefix_ids + x_ids + suffix_ids
    input_ids = prompt_ids + y_ids
    attention_mask = [1] * len(input_ids)

    labels = [-100] * len(prompt_ids) + y_ids

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


In [ ]:
train_dataset = train_dataset.map(
    format_example,
    remove_columns=train_dataset.column_names,
)

test_dataset = test_dataset.map(
    format_example,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/719 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator

training_args = TrainingArguments(
    output_dir="./llama32_1b_robotics_lora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_strategy="steps",
    logging_steps=1,
    logging_first_step=True,
    save_steps=200,
    eval_steps=200,
    eval_strategy="steps",
    save_strategy="steps",
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=default_data_collator,
)

trainer.train()

trainer.model.save_pretrained("./llama32_1b_robotics_lora/final_adapter")
tokenizer.save_pretrained("./llama32_1b_robotics_lora/final_adapter")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
180,2.242079,2.028395


('./llama32_1b_robotics_lora/final_adapter/tokenizer_config.json',
 './llama32_1b_robotics_lora/final_adapter/chat_template.jinja',
 './llama32_1b_robotics_lora/final_adapter/tokenizer.json')

In [ ]:
!zip -r /content/file.zip /content/llama32_1b_robotics_lora/

  adding: content/llama32_1b_robotics_lora/ (stored 0%)
  adding: content/llama32_1b_robotics_lora/checkpoint-180/ (stored 0%)
  adding: content/llama32_1b_robotics_lora/checkpoint-180/training_args.bin (deflated 53%)
  adding: content/llama32_1b_robotics_lora/checkpoint-180/README.md (deflated 65%)
  adding: content/llama32_1b_robotics_lora/checkpoint-180/adapter_model.safetensors (deflated 7%)
  adding: content/llama32_1b_robotics_lora/checkpoint-180/trainer_state.json (deflated 76%)
  adding: content/llama32_1b_robotics_lora/checkpoint-180/rng_state.pth (deflated 26%)
  adding: content/llama32_1b_robotics_lora/checkpoint-180/scheduler.pt (deflated 62%)
  adding: content/llama32_1b_robotics_lora/checkpoint-180/adapter_config.json (deflated 58%)
  adding: content/llama32_1b_robotics_lora/checkpoint-180/optimizer.pt (deflated 11%)
  adding: content/llama32_1b_robotics_lora/final_adapter/ (stored 0%)
  adding: content/llama32_1b_robotics_lora/final_adapter/README.md (deflated 65%)
  add